## SRP074490

**paper:** [PMID: 27230888](https://pmc.ncbi.nlm.nih.gov/articles/PMC4882810/) - Sixteen kiwi (Apteryx spp) transcriptomes provide a wealth of genetic markers and insight into sex chromosome evolution in birds, 2016

**date, curator:** 2026-03-20, Sara Carsanaro

**notes**
* for dev stage: 
    * All LSK were adults when sampled apart from a single juvenile male from Kapiti Island.
        * not clear which animals were sampled where, so all male LSKs were noted as fully formed (juvenile or adult) and all females were noted as adults
    * We sampled juvenile rowi prior to breeding age and in the absence of predators or adult rowi
        * all Rowi were noted as juvenile

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Blood,UBERON:0000178,blood,perfect match


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,fully formed,UBERON:0000066,fully formed stage,perfect match
2,adult,UBERON:0000113,post-juvenile adult stage,perfect match
14,juvenile,UBERON:0034919,juvenile stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP074490"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 16/16 [00:17<00:00,  1.10s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1755870,SRP074490,Illumina HiSeq 2000,SRS1424433,,,,,,Blood,fully formed,,,,M,,,8824,,,,,1,LSK3,SAMN04958704,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
1,SRX1745537,SRP074490,Illumina HiSeq 2000,SRS1424433,,,,,,Blood,fully formed,,,,M,,,8824,,,,,1,LSK3,SAMN04958704,,,,,,,,,,,20/03/2026,NA,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
2,SRX1755873,SRP074490,Illumina HiSeq 2000,SRS1424486,,,,,,Blood,adult,,,,F,,,8824,,,,,2,LSK6,SAMN04958802,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK6,,,,,,TRANSCRIPTOMIC,RT-PCR
3,SRX1745590,SRP074490,Illumina HiSeq 2000,SRS1424486,,,,,,Blood,adult,,,,F,,,8824,,,,,2,LSK6,SAMN04958802,,,,,,,,,,,20/03/2026,NA,,LSK6,,,,,,TRANSCRIPTOMIC,other
4,SRX1755875,SRP074490,Illumina HiSeq 2000,SRS1424559,,,,,,Blood,fully formed,,,,M,,,8824,,,,,3,LSK8,SAMN04958807,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK8,,,,,,TRANSCRIPTOMIC,RT-PCR
5,SRX1745686,SRP074490,Illumina HiSeq 2000,SRS1424559,,,,,,Blood,fully formed,,,,M,,,8824,,,,,3,LSK8,SAMN04958807,,,,,,,,,,,20/03/2026,NA,,LSK8,,,,,,TRANSCRIPTOMIC,other
6,SRX1755872,SRP074490,Illumina HiSeq 2000,SRS1424560,,,,,,Blood,adult,,,,F,,,8824,,,,,4,LSK5,SAMN04958809,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK5,,,,,,TRANSCRIPTOMIC,RT-PCR
7,SRX1745697,SRP074490,Illumina HiSeq 2000,SRS1424560,,,,,,Blood,adult,,,,F,,,8824,,,,,4,LSK5,SAMN04958809,,,,,,,,,,,20/03/2026,NA,,LSK5,,,,,,TRANSCRIPTOMIC,other
8,SRX1755871,SRP074490,Illumina HiSeq 2000,SRS1426043,,,,,,Blood,fully formed,,,,M,,,8824,,,,,5,LSK4,SAMN04958825,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK4,,,,,,TRANSCRIPTOMIC,RT-PCR
9,SRX1748254,SRP074490,Illumina HiSeq 2000,SRS1426043,,,,,,Blood,fully formed,,,,M,,,8824,,,,,5,LSK4,SAMN04958825,,,,,,,,,,,20/03/2026,NA,,LSK4,,,,,,TRANSCRIPTOMIC,other


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Blood']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000178'
library.loc[:,'anatName'] = 'blood'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1755870,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,,,,Blood,fully formed,perfect match,not documented,,M,,,8824,,,,,1,LSK3,SAMN04958704,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
1,SRX1745537,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,,,,Blood,fully formed,perfect match,not documented,,M,,,8824,,,,,1,LSK3,SAMN04958704,,,,,,,,,,,20/03/2026,NA,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
2,SRX1755873,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,,,,Blood,adult,perfect match,not documented,,F,,,8824,,,,,2,LSK6,SAMN04958802,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK6,,,,,,TRANSCRIPTOMIC,RT-PCR
3,SRX1745590,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,,,,Blood,adult,perfect match,not documented,,F,,,8824,,,,,2,LSK6,SAMN04958802,,,,,,,,,,,20/03/2026,NA,,LSK6,,,,,,TRANSCRIPTOMIC,other
4,SRX1755875,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,,,,Blood,fully formed,perfect match,not documented,,M,,,8824,,,,,3,LSK8,SAMN04958807,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK8,,,,,,TRANSCRIPTOMIC,RT-PCR
5,SRX1745686,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,,,,Blood,fully formed,perfect match,not documented,,M,,,8824,,,,,3,LSK8,SAMN04958807,,,,,,,,,,,20/03/2026,NA,,LSK8,,,,,,TRANSCRIPTOMIC,other
6,SRX1755872,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,,,,Blood,adult,perfect match,not documented,,F,,,8824,,,,,4,LSK5,SAMN04958809,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK5,,,,,,TRANSCRIPTOMIC,RT-PCR
7,SRX1745697,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,,,,Blood,adult,perfect match,not documented,,F,,,8824,,,,,4,LSK5,SAMN04958809,,,,,,,,,,,20/03/2026,NA,,LSK5,,,,,,TRANSCRIPTOMIC,other
8,SRX1755871,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,,,,Blood,fully formed,perfect match,not documented,,M,,,8824,,,,,5,LSK4,SAMN04958825,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK4,,,,,,TRANSCRIPTOMIC,RT-PCR
9,SRX1748254,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,,,,Blood,fully formed,perfect match,not documented,,M,,,8824,,,,,5,LSK4,SAMN04958825,,,,,,,,,,,20/03/2026,NA,,LSK4,,,,,,TRANSCRIPTOMIC,other


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['adult' 'fully formed' 'juvenile']


In [8]:
# adult
library.loc[library["infoStage"] == "adult", "stageId"] = "UBERON:0000113"
library.loc[library["infoStage"] == "adult", "stageName"] = "post-juvenile adult stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "adult", "stageAnnotationStatus"] = "perfect match"

# fully formed
library.loc[library["infoStage"] == "fully formed", "stageId"] = "UBERON:0000066"
library.loc[library["infoStage"] == "fully formed", "stageName"] = "fully formed stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "fully formed", "stageAnnotationStatus"] = "perfect match"

# juvenile
library.loc[library["infoStage"] == "juvenile", "stageId"] = "UBERON:0034919"
library.loc[library["infoStage"] == "juvenile", "stageName"] = "juvenile stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "juvenile", "stageAnnotationStatus"] = "perfect match"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1755870,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,,,,,1,LSK3,SAMN04958704,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
1,SRX1745537,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,,,,,1,LSK3,SAMN04958704,,,,,,,,,,,20/03/2026,NA,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
2,SRX1755873,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,,,,,2,LSK6,SAMN04958802,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK6,,,,,,TRANSCRIPTOMIC,RT-PCR
3,SRX1745590,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,,,,,2,LSK6,SAMN04958802,,,,,,,,,,,20/03/2026,NA,,LSK6,,,,,,TRANSCRIPTOMIC,other
4,SRX1755875,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,,,,,3,LSK8,SAMN04958807,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK8,,,,,,TRANSCRIPTOMIC,RT-PCR
5,SRX1745686,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,,,,,3,LSK8,SAMN04958807,,,,,,,,,,,20/03/2026,NA,,LSK8,,,,,,TRANSCRIPTOMIC,other
6,SRX1755872,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,,,,,4,LSK5,SAMN04958809,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK5,,,,,,TRANSCRIPTOMIC,RT-PCR
7,SRX1745697,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,,,,,4,LSK5,SAMN04958809,,,,,,,,,,,20/03/2026,NA,,LSK5,,,,,,TRANSCRIPTOMIC,other
8,SRX1755871,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,,,,,5,LSK4,SAMN04958825,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK4,,,,,,TRANSCRIPTOMIC,RT-PCR
9,SRX1748254,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,,,,,5,LSK4,SAMN04958825,,,,,,,,,,,20/03/2026,NA,,LSK4,,,,,,TRANSCRIPTOMIC,other


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq RNA Sample Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1755870,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,LSK3,SAMN04958704,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
1,SRX1745537,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,LSK3,SAMN04958704,,,,,,,,,,,20/03/2026,NA,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
2,SRX1755873,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,LSK6,SAMN04958802,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK6,,,,,,TRANSCRIPTOMIC,RT-PCR
3,SRX1745590,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,LSK6,SAMN04958802,,,,,,,,,,,20/03/2026,NA,,LSK6,,,,,,TRANSCRIPTOMIC,other
4,SRX1755875,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,LSK8,SAMN04958807,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK8,,,,,,TRANSCRIPTOMIC,RT-PCR
5,SRX1745686,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,LSK8,SAMN04958807,,,,,,,,,,,20/03/2026,NA,,LSK8,,,,,,TRANSCRIPTOMIC,other
6,SRX1755872,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,4,LSK5,SAMN04958809,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK5,,,,,,TRANSCRIPTOMIC,RT-PCR
7,SRX1745697,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,4,LSK5,SAMN04958809,,,,,,,,,,,20/03/2026,NA,,LSK5,,,,,,TRANSCRIPTOMIC,other
8,SRX1755871,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,5,LSK4,SAMN04958825,,,,,,,,,,,20/03/2026,Random sequencing of whole transcriptome,,LSK4,,,,,,TRANSCRIPTOMIC,RT-PCR
9,SRX1748254,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,5,LSK4,SAMN04958825,,,,,,,,,,,20/03/2026,NA,,LSK4,,,,,,TRANSCRIPTOMIC,other


#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

    #libraryId       SRSId
0   SRX1755870  SRS1424433
1   SRX1745537  SRS1424433
2   SRX1755873  SRS1424486
3   SRX1745590  SRS1424486
4   SRX1755875  SRS1424559
5   SRX1745686  SRS1424559
6   SRX1755872  SRS1424560
7   SRX1745697  SRS1424560
8   SRX1755871  SRS1426043
9   SRX1748254  SRS1426043
10  SRX1755874  SRS1426044
11  SRX1748253  SRS1426044
12  SRX1755863  SRS1426045
13  SRX1748255  SRS1426045
15  SRX1748257  SRS1426046
14  SRX1755864  SRS1426046
16  SRX1755876  SRS1426047
17  SRX1748252  SRS1426047
18  SRX1755866  SRS1426048
19  SRX1748256  SRS1426048
20  SRX1755865  SRS1426049
21  SRX1748258  SRS1426049
22  SRX1755869  SRS1426050
23  SRX1748259  SRS1426050
24  SRX1755877  SRS1426051
25  SRX1748261  SRS1426051
26  SRX1755867  SRS1426052
27  SRX1748260  SRS1426052
28  SRX1755868  SRS1426054
29  SRX1748262  SRS1426054


/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_1982/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-20'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX1755870,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,LSK3,SAMN04958704,,,,,,,,,,SAC,2026-03-20,Random sequencing of whole transcriptome,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
1,SRX1745537,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,LSK3,SAMN04958704,,,,,,,,,,SAC,2026-03-20,NA,,LSK3,,,,,,TRANSCRIPTOMIC,RT-PCR
2,SRX1755873,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,LSK6,SAMN04958802,,,,,,,,,,SAC,2026-03-20,Random sequencing of whole transcriptome,,LSK6,,,,,,TRANSCRIPTOMIC,RT-PCR
3,SRX1745590,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,LSK6,SAMN04958802,,,,,,,,,,SAC,2026-03-20,NA,,LSK6,,,,,,TRANSCRIPTOMIC,other
4,SRX1755875,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,LSK8,SAMN04958807,,,,,,,,,,SAC,2026-03-20,Random sequencing of whole transcriptome,,LSK8,,,,,,TRANSCRIPTOMIC,RT-PCR
5,SRX1745686,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,LSK8,SAMN04958807,,,,,,,,,,SAC,2026-03-20,NA,,LSK8,,,,,,TRANSCRIPTOMIC,other
6,SRX1755872,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,4,LSK5,SAMN04958809,,,,,,,,,,SAC,2026-03-20,Random sequencing of whole transcriptome,,LSK5,,,,,,TRANSCRIPTOMIC,RT-PCR
7,SRX1745697,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,4,LSK5,SAMN04958809,,,,,,,,,,SAC,2026-03-20,NA,,LSK5,,,,,,TRANSCRIPTOMIC,other
8,SRX1755871,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,5,LSK4,SAMN04958825,,,,,,,,,,SAC,2026-03-20,Random sequencing of whole transcriptome,,LSK4,,,,,,TRANSCRIPTOMIC,RT-PCR
9,SRX1748254,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,5,LSK4,SAMN04958825,,,,,,,,,,SAC,2026-03-20,NA,,LSK4,,,,,,TRANSCRIPTOMIC,other


#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 27230888'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX1755870,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,LSK3,SAMN04958704,,,,,,,PMID: 27230888,,,SAC,2026-03-20
1,SRX1745537,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,LSK3,SAMN04958704,,,,,,,PMID: 27230888,,,SAC,2026-03-20
2,SRX1755873,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,LSK6,SAMN04958802,,,,,,,PMID: 27230888,,,SAC,2026-03-20
3,SRX1745590,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,LSK6,SAMN04958802,,,,,,,PMID: 27230888,,,SAC,2026-03-20
4,SRX1755875,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,LSK8,SAMN04958807,,,,,,,PMID: 27230888,,,SAC,2026-03-20
5,SRX1745686,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,LSK8,SAMN04958807,,,,,,,PMID: 27230888,,,SAC,2026-03-20
6,SRX1755872,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,4,LSK5,SAMN04958809,,,,,,,PMID: 27230888,,,SAC,2026-03-20
7,SRX1745697,SRP074490,Illumina HiSeq 2000,SRS1424560,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,4,LSK5,SAMN04958809,,,,,,,PMID: 27230888,,,SAC,2026-03-20
8,SRX1755871,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,5,LSK4,SAMN04958825,,,,,,,PMID: 27230888,,,SAC,2026-03-20
9,SRX1748254,SRP074490,Illumina HiSeq 2000,SRS1426043,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,5,LSK4,SAMN04958825,,,,,,,PMID: 27230888,,,SAC,2026-03-20


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP074490,Sixteen kiwi (Apteryx spp) transcriptomes provide a wealth of genetic markers and insight into sex chromosome evolution in birds,Raw data reads from the transcriptome of 16 individuals (8 from Apertyxowenii and 8 from Apertyx rowi).,SRA,,,,,,,PRJNA320876,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

31

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP074490,Sixteen kiwi (Apteryx spp) transcriptomes provide a wealth of genetic markers and insight into sex chromosome evolution in birds,Raw data reads from the transcriptome of 16 individuals (8 from Apertyxowenii and 8 from Apertyx rowi).,SRA,total,Bgee 1K,31,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA320876,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '27230888'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4882810/'
experiment.loc[:,'DOI'] = '10.1186/s12864-016-2714-2'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP074490,Sixteen kiwi (Apteryx spp) transcriptomes provide a wealth of genetic markers and insight into sex chromosome evolution in birds,Raw data reads from the transcriptome of 16 individuals (8 from Apertyxowenii and 8 from Apertyx rowi).,SRA,total,Bgee 1K,31,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA320876,27230888,https://pmc.ncbi.nlm.nih.gov/articles/PMC4882810/,10.1186/s12864-016-2714-2,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
56351,SRX736627,SRP049083,Illumina HiSeq 2000,SRS724683,UBERON:0002037,cerebellum,UBERON:0000113,post-juvenile adult stage,,Cerebellum,adult,perfect match,not documented,perfect match,M,,,8801,TruSeq RNA Sample Preparation Kit,full_length,polyA,,,Struthio camelus cerebellum RNA-Seq,SAMN03121476,,,,,,,PMID: 26199010,,,SAC,2026-03-19
56352,SRX1092592,SRP060404,Illumina HiSeq 2000,SRS988177,UBERON:0002370,thymus,UBERON:0000112,sexually immature stage,,thymus,90-days,perfect match,not documented,missing child term,NA,,,8801,,,polyA,,,Model organism or animal sample from Struthio ...,SAMN03840799,90,day,,,,,PMID: 30094741,,,SAC,2026-03-19
56353,SRX1755870,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,LSK3,SAMN04958704,,,,,,,PMID: 27230888,,,SAC,2026-03-20
56354,SRX1745537,SRP074490,Illumina HiSeq 2000,SRS1424433,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,1,LSK3,SAMN04958704,,,,,,,PMID: 27230888,,,SAC,2026-03-20
56355,SRX1755873,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,LSK6,SAMN04958802,,,,,,,PMID: 27230888,,,SAC,2026-03-20
56356,SRX1745590,SRP074490,Illumina HiSeq 2000,SRS1424486,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Blood,adult,perfect match,not documented,perfect match,F,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,2,LSK6,SAMN04958802,,,,,,,PMID: 27230888,,,SAC,2026-03-20
56357,SRX1755875,SRP074490,Illumina HiSeq 2000,SRS1424559,UBERON:0000178,blood,UBERON:0000066,fully formed stage,,Blood,fully formed,perfect match,not documented,perfect match,M,,,8824,TruSeq RNA Sample Preparation Kit,full_length,polyA,,3,LSK8,SAMN04958807,,,,,,,PMID: 27230888,,,SAC,2026-03-20


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1101,SRP049083,Struthio camelus Transcriptome,We sequenced the transcriptome from Struthio c...,SRA,total,Bgee 1K,2,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA264356,26199010,https://pmc.ncbi.nlm.nih.gov/articles/PMC4509753/,10.1186/s12864-015-1769-9,,
1102,SRP060404,Struthio camelus camelus transcriptome or Gene...,The goal of this project is to identify the me...,SRA,partial,Bgee 1K,1,,,,PRJNA287589,30094741,https://link.springer.com/article/10.1007/s120...,10.1007/s12011-018-1441-8,,removed libraries given boron to cause immune ...
1103,SRP074490,Sixteen kiwi (Apteryx spp) transcriptomes prov...,Raw data reads from the transcriptome of 16 in...,SRA,total,Bgee 1K,31,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA320876,27230888,https://pmc.ncbi.nlm.nih.gov/articles/PMC4882810/,10.1186/s12864-016-2714-2,,


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [27]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 4541ddd] adding annotated bulk experiment SRP074490
 2 files changed, 32 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.33 KiB | 2.33 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   a65ea1b..4541ddd  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push